In [6]:
# =============================================================================
# MACROECONOMICS PROJECT: US MARKET SPILLOVERS TO INTERNATIONAL MARKETS
# Clean Implementation
# =============================================================================

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

#Our 8 ETFs
developed_etfs = ['EWU', 'EWG', 'EWJ', 'EWA']  # UK, Germany, Japan, Australia
emerging_etfs = ['EWZ', 'EWW', 'EZA']   # Brazil, Mexico, South Africa
all_etfs = developed_etfs + emerging_etfs

# Control variables
controls = ['^VIX', 'CL=F', 'DX-Y.NYB', '^TNX']  # VIX, Oil, Dollar Index, 10Y Treasury


# Get ETF data from 2000 to present
etf_data = yf.download(all_etfs, start='2003-02-16', end='2025-12-31', progress=False)


# Get control variables
control_data = yf.download(controls, start='2003-02-16', end='2025-12-31', progress=False)


# US market (S&P 500 for US disruption identification)
us_market = yf.download('^GSPC', start='2003-02-16', end='2025-12-31', progress=False)

print("✅ Data download complete!")

# Calculate daily returns for all our variables

# ETF returns
etf_returns = etf_data['Close'].pct_change() * 100  # Convert to percentage
etf_returns.dropna(inplace=True)

# US market returns
us_returns = us_market['Close'].pct_change() * 100
us_returns.dropna(inplace=True)
us_returns.name = 'SPX_return'

# Control variable returns/changes
control_returns = control_data['Close'].pct_change() * 100
control_returns.dropna(inplace=True)

print("Quick data preview:")
print(f"ETF returns (last 5 days):")
print(etf_returns.tail())

print(f"\nUS market returns (last 5 days):")
print(us_returns.tail())

print(f"\nControl variables (last 5 days):")
print(control_returns.tail())

print(f"Sample period: {etf_returns.index.min()} to {etf_returns.index.max()}")
print(f"Total observations: {len(etf_returns)}")

✅ Data download complete!
Quick data preview:
ETF returns (last 5 days):
Ticker           EWA       EWG       EWJ       EWU       EWW       EWZ  \
Date                                                                     
2025-12-23  1.759086  0.236180  0.845141  0.433886  1.558262  2.195673   
2025-12-24 -0.037581  0.306317 -0.172541  0.227381 -0.126695 -0.315956   
2025-12-26  0.300752  0.187930 -0.037036  0.362976  0.281895  0.570526   
2025-12-29 -0.749628 -0.328261  0.148197 -0.497290 -0.899507 -0.976994   
2025-12-30 -0.264349  0.564577 -0.135643  0.454341 -1.446609  2.158775   

Ticker           EZA  
Date                  
2025-12-23  1.090275  
2025-12-24  0.546444  
2025-12-26  0.557779  
2025-12-29 -2.673869  
2025-12-30  1.169082  

US market returns (last 5 days):
Ticker         ^GSPC
Date                
2025-12-23  0.455039
2025-12-24  0.322148
2025-12-26 -0.030436
2025-12-29 -0.349205
2025-12-30 -0.137567

Control variables (last 5 days):
Ticker          CL=F  DX-Y.NYB  

In [13]:
# Check data availability by ETF
for etf in ['EWA', 'EWG', 'EWJ', 'EWU', 'EWW', 'EWZ', 'EZA']:
    etf_data = regression_data[etf].dropna()
    print(f"{etf}: {len(etf_data)} observations, range: {etf_data.index.min()} to {etf_data.index.max()}")

# Add this debugging code to see what's being filtered out
def debug_regression_data(etf_name):
    print(f"\n🔍 Debugging {etf_name}:")
    
    # Check the original ETF data
    etf_series = regression_data[etf_name]
    print(f"Total {etf_name} observations: {len(etf_series)}")
    print(f"Non-null {etf_name} observations: {etf_series.notna().sum()}")
    print(f"Null {etf_name} observations: {etf_series.isna().sum()}")
    
    # Check for infinite values
    print(f"Infinite values in {etf_name}: {np.isinf(etf_series).sum()}")
    
    # Check the required variables
    required_vars = ['us_return', 'event_dummy', 'vix_change', 'oil_change', 'dollar_change']
    for var in required_vars:
        if var in regression_data.columns:
            var_series = regression_data[var]
            print(f"{var} - Total: {len(var_series)}, Non-null: {var_series.notna().sum()}")
    
    # Check what remains after dropna() on all required columns
    all_vars = [etf_name] + required_vars
    clean_data = regression_data[all_vars].dropna()
    print(f"Observations after dropna() on all variables: {len(clean_data)}")
    
    return clean_data

# Test with one ETF
test_data = debug_regression_data('EWA')
print("\nFirst 5 rows of clean data:")
print(test_data.head())

print(regression_data[['pre_event', 'post_event']].isna().sum())
print(regression_data[['pre_event', 'post_event']].head())


EWA: 5754 observations, range: 2003-02-19 00:00:00 to 2025-12-30 00:00:00
EWG: 5754 observations, range: 2003-02-19 00:00:00 to 2025-12-30 00:00:00
EWJ: 5754 observations, range: 2003-02-19 00:00:00 to 2025-12-30 00:00:00
EWU: 5754 observations, range: 2003-02-19 00:00:00 to 2025-12-30 00:00:00
EWW: 5754 observations, range: 2003-02-19 00:00:00 to 2025-12-30 00:00:00
EWZ: 5754 observations, range: 2003-02-19 00:00:00 to 2025-12-30 00:00:00
EZA: 5754 observations, range: 2003-02-19 00:00:00 to 2025-12-30 00:00:00

🔍 Debugging EWA:
Total EWA observations: 5754
Non-null EWA observations: 5754
Null EWA observations: 0
Infinite values in EWA: 0
us_return - Total: 5754, Non-null: 5754
event_dummy - Total: 5754, Non-null: 5754
vix_change - Total: 5754, Non-null: 5754
oil_change - Total: 5754, Non-null: 5754
dollar_change - Total: 5754, Non-null: 5754
Observations after dropna() on all variables: 5754

First 5 rows of clean data:
                 EWA  us_return  event_dummy  vix_change  oil_ch

In [8]:
# =============================================================================
# STEP 2: EVENT IDENTIFICATION - CLEAN APPROACH
# =============================================================================

def identify_us_events(us_returns, threshold=8.0):
    """
    Identify major US market disruption events
    """
    # Handle both Series and DataFrame
    if isinstance(us_returns, pd.DataFrame):
        us_col = us_returns.columns[0]
        daily_returns = us_returns[us_col]
    else:
        daily_returns = us_returns
    
    # Find extreme moves
    extreme_events = daily_returns[abs(daily_returns) > threshold]
    
    print(f"📊 EVENT IDENTIFICATION RESULTS:")
    print(f"Total events with |return| > {threshold}%: {len(extreme_events)}")
    print(f"Crash events (< -{threshold}%): {len(extreme_events[extreme_events < 0])}")
    print(f"Surge events (> {threshold}%): {len(extreme_events[extreme_events > 0])}")
    
    return extreme_events.index.tolist()

# Identify events
event_dates = identify_us_events(us_returns, threshold=8.0)

print(f"\n🎯 IDENTIFIED EVENTS:")
for date in sorted(event_dates)[-10:]:  # Show last 10 events
    if isinstance(us_returns, pd.DataFrame):
        ret_val = us_returns.loc[date, us_returns.columns[0]]
    else:
        ret_val = us_returns.loc[date]
    print(f"{date.strftime('%Y-%m-%d')}: {ret_val:.2f}%")

📊 EVENT IDENTIFICATION RESULTS:
Total events with |return| > 8.0%: 10
Crash events (< -8.0%): 5
Surge events (> 8.0%): 5

🎯 IDENTIFIED EVENTS:
2008-09-29: -8.81%
2008-10-13: 11.58%
2008-10-15: -9.03%
2008-10-28: 10.79%
2008-12-01: -8.93%
2020-03-12: -9.51%
2020-03-13: 9.29%
2020-03-16: -11.98%
2020-03-24: 9.38%
2025-04-09: 9.52%


In [9]:
# =============================================================================
# STEP 3: CREATE EVENT WINDOWS AND DUMMY VARIABLES (CONTINUED)
# =============================================================================

def create_event_variables(returns_data, event_dates, window=[-5, 10]):
    """
    Create event dummy variables for regression analysis
    """
    # Create base dataframe with all dates
    analysis_df = pd.DataFrame(index=returns_data.index)
    
    # Event dummy (1 on event days, 0 otherwise)
    analysis_df['event_dummy'] = 0
    analysis_df.loc[analysis_df.index.isin(event_dates), 'event_dummy'] = 1
    
    # Event window dummy (1 for days within window around events)
    analysis_df['event_window'] = 0
    
    for event_date in event_dates:
        start_date = event_date + pd.Timedelta(days=window[0])
        end_date = event_date + pd.Timedelta(days=window[1])
        
        # Mark all days in the window
        mask = (analysis_df.index >= start_date) & (analysis_df.index <= end_date)
        analysis_df.loc[mask, 'event_window'] = 1
    
    # Pre-event and post-event dummies for robustness
    analysis_df['pre_event'] = 0
    analysis_df['post_event'] = 0
    
    for event_date in event_dates:
        pre_start = event_date + pd.Timedelta(days=-5)
        pre_end = event_date + pd.Timedelta(days=-1)
        post_start = event_date + pd.Timedelta(days=1)
        post_end = event_date + pd.Timedelta(days=10)
        
        pre_mask = (analysis_df.index >= pre_start) & (analysis_df.index <= pre_end)
        post_mask = (analysis_df.index >= post_start) & (analysis_df.index <= post_end)
        
        analysis_df.loc[pre_mask, 'pre_event'] = 1
        analysis_df.loc[post_mask, 'post_event'] = 1
    
    return analysis_df

# Create event variables
event_vars = create_event_variables(etf_returns, event_dates)

print(f"📈 EVENT VARIABLES CREATED:")
print(f"Event days: {event_vars['event_dummy'].sum()}")
print(f"Event window days: {event_vars['event_window'].sum()}")
print(f"Pre-event days: {event_vars['pre_event'].sum()}")
print(f"Post-event days: {event_vars['post_event'].sum()}")


📈 EVENT VARIABLES CREATED:
Event days: 10
Event window days: 74
Pre-event days: 24
Post-event days: 55


In [10]:
# =============================================================================
# STEP 4: PREPARE REGRESSION DATASET (CONTINUED)
# =============================================================================

def prepare_regression_data(etf_returns, event_vars, control_returns, us_returns):
    """
    Prepare the final dataset for regression analysis
    """
    # Start with ETF returns
    reg_data = etf_returns.copy()
    
    # Add US market returns
    if isinstance(us_returns, pd.DataFrame):
        us_col = us_returns.columns[0]
        reg_data['us_return'] = us_returns[us_col]
    else:
        reg_data['us_return'] = us_returns
    
    # Add event variables
    reg_data = reg_data.join(event_vars, how='inner')
    
    # Add control variables (handle potential missing data)
    if control_returns is not None and not control_returns.empty:
        # Select key controls and rename for clarity
        controls_clean = control_returns.copy()
        if '^VIX' in controls_clean.columns:
            reg_data['vix_change'] = controls_clean['^VIX']
        if 'CL=F' in controls_clean.columns:
            reg_data['oil_change'] = controls_clean['CL=F']
        if 'DX-Y.NYB' in controls_clean.columns:
            reg_data['dollar_change'] = controls_clean['DX-Y.NYB']
    
    # Remove any rows with missing data
    reg_data = reg_data.dropna()
    
    return reg_data

# Prepare the regression dataset
regression_data = prepare_regression_data(etf_returns, event_vars, control_returns, us_returns)

print(f"📊 REGRESSION DATASET PREPARED:")
print(f"Observations: {len(regression_data)}")
print(f"Variables: {list(regression_data.columns)}")
print(f"Date range: {regression_data.index.min()} to {regression_data.index.max()}")


📊 REGRESSION DATASET PREPARED:
Observations: 5754
Variables: ['EWA', 'EWG', 'EWJ', 'EWU', 'EWW', 'EWZ', 'EZA', 'us_return', 'event_dummy', 'event_window', 'pre_event', 'post_event', 'vix_change', 'oil_change', 'dollar_change']
Date range: 2003-02-19 00:00:00 to 2025-12-30 00:00:00


In [14]:
# =============================================================================
# STEP 5: RUN MARKET MODEL REGRESSION (WITH VARIABLE DEFINITIONS)
# =============================================================================

import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_white
from statsmodels.stats import diagnostic

# Define ETF tickers (assuming these are the ones we're analyzing)
etf_tickers = ['EWA', 'EWG', 'EWJ', 'EWU', 'EWW', 'EWZ', 'EZA']  # Adjust based on your actual ETFs

def run_market_model_regression(data, etf_col):
    """
    Run market model regression with event dummies
    """
    # Check if ETF column exists in data
    if etf_col not in data.columns:
        print(f"Warning: {etf_col} not found in data")
        return None, False
    
    # Prepare dependent variable
    y = data[etf_col]
    
    # Prepare independent variables
    X = pd.DataFrame(index=data.index)
    X['us_return'] = data['us_return']
    X['event_dummy'] = data['event_dummy']
    X['pre_event'] = data['pre_event']
    X['post_event'] = data['post_event']
    
    # Add control variables if available
    control_vars = ['vix_change', 'oil_change', 'dollar_change']
    for var in control_vars:
        if var in data.columns:
            X[var] = data[var]
    
     # Add constant correctly
    X = sm.add_constant(X)  # or: X['constant'] = 1 after X is created with proper index

    # Remove any rows with NaN values
    combined_data = pd.concat([y, X], axis=1).dropna()

    y_clean = combined_data[etf_col]
    X_clean = combined_data.drop(columns=[etf_col])
    
    if len(y_clean) == 0:
        print(f"Warning: No valid data for {etf_col}")
        return None, False
    
    # Run regression
    try:
        model = sm.OLS(y_clean, X_clean).fit()
        
        # Test for heteroscedasticity
        _, pvalue_het, _, _ = het_white(model.resid, model.model.exog)
        
        # Use robust standard errors if heteroscedasticity detected
        if pvalue_het < 0.05:
            model_robust = sm.OLS(y_clean, X_clean).fit(cov_type='HC3')
            return model_robust, True
        else:
            return model, False
            
    except Exception as e:
        print(f"Error running regression for {etf_col}: {e}")
        return None, False

# Get actual ETF columns from the regression data
actual_etf_cols = [col for col in regression_data.columns if col in all_etfs]
    
print(f"📊 Available ETF columns: {actual_etf_cols}")

# Run regression for each ETF
regression_results = {}
robust_se_used = {}

for etf in actual_etf_cols:
    print("🔍 Checking regression_data columns:")
    print(regression_data.columns.tolist())
    print(f"🔍 Sample of regression_data:")
    print(regression_data.head())
    print(f"🔍 actual_etf_cols found: {actual_etf_cols}")
    print(f"\n🔄 Processing {etf}...")
    model, robust = run_market_model_regression(regression_data, etf)
    
    if model is not None:
        regression_results[etf] = model
        robust_se_used[etf] = robust
        
        print(f"📈 REGRESSION RESULTS FOR {etf}:")
        print(f"Robust SE used: {robust}")
        print(f"R-squared: {model.rsquared:.4f}")
        print(f"Observations: {model.nobs}")
        
        # Check if event_dummy coefficient exists
        if 'event_dummy' in model.params:
            print(f"Event coefficient: {model.params['event_dummy']:.6f}")
            print(f"Event p-value: {model.pvalues['event_dummy']:.4f}")
            if model.pvalues['event_dummy'] < 0.05:
                print("🔴 SIGNIFICANT EVENT EFFECT DETECTED!")
        
        # Display key coefficients
        key_coeffs = ['event_dummy', 'pre_event', 'post_event', 'us_return']
        for coeff in key_coeffs:
            if coeff in model.params:
                print(f"{coeff}: {model.params[coeff]:.6f} (p={model.pvalues[coeff]:.4f})")
    else:
        print(f"❌ Failed to run regression for {etf}")

print(f"\n✅ Completed regressions for {len(regression_results)} ETFs")


📊 Available ETF columns: ['EWA', 'EWG', 'EWJ', 'EWU', 'EWW', 'EWZ', 'EZA']
🔍 Checking regression_data columns:
['EWA', 'EWG', 'EWJ', 'EWU', 'EWW', 'EWZ', 'EZA', 'us_return', 'event_dummy', 'event_window', 'pre_event', 'post_event', 'vix_change', 'oil_change', 'dollar_change', 'event_pos', 'event_neg']
🔍 Sample of regression_data:
                 EWA       EWG       EWJ       EWU       EWW       EWZ  \
Date                                                                     
2003-02-19  0.767581 -1.783828 -1.687737 -1.822857 -1.444347 -1.866691   
2003-02-20 -1.088144 -0.106836  0.000000  0.088392  0.517254  0.135842   
2003-02-21  0.219996  0.106951  0.143049  0.088324  0.428821  0.000000   
2003-02-24  1.207485 -2.243588 -0.285679 -0.353044 -0.426989  1.628257   
2003-02-25  0.216936 -1.092866 -1.432691 -0.708599 -1.457999 -2.269674   

                 EZA  us_return  event_dummy  event_window  pre_event  \
Date                                                                    
200

In [19]:
# =============================================================================
# STEP 5.5: RUN MARKET MODEL REGRESSION (WITH VARIABLE DEFINITIONS)
# Changed regression set up to distinguish between positive and negative effects
# =============================================================================

import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_white
from statsmodels.stats import diagnostic

# Define ETF tickers (assuming these are the ones we're analyzing)
etf_tickers = ['EWA', 'EWG', 'EWJ', 'EWU', 'EWW', 'EWZ', 'EZA']  # Adjust based on your actual ETFs

# Initialize with 0
regression_data['event_pos'] = 0
regression_data['event_neg'] = 0

# Set 1 on the appropriate dates
for _, row in event_df.iterrows():
    d = row['event_date']
    if d in regression_data.index:
        if row['type'] == 'positive':
            regression_data.loc[d, 'event_pos'] = 1
        else:
            regression_data.loc[d, 'event_neg'] = 1
            
def run_market_model_regression(data, etf_col):
    """
    Run market model regression with event dummies
    """
    # Check if ETF column exists in data
    if etf_col not in data.columns:
        print(f"Warning: {etf_col} not found in data")
        return None, False
    
    # Prepare dependent variable
    y = data[etf_col]
    
    # Prepare independent variables
    X = pd.DataFrame(index=data.index)
    X['us_return'] = data['us_return']
    X['pre_event'] = data['pre_event']
    X['post_event'] = data['post_event']
    X['event_pos'] = data['event_pos']
    X['event_neg'] = data['event_neg']
    
    # Add control variables if available
    control_vars = ['vix_change', 'oil_change', 'dollar_change']
    for var in control_vars:
        if var in data.columns:
            X[var] = data[var]
    
     # Add constant correctly
    X = sm.add_constant(X)  # or: X['constant'] = 1 after X is created with proper index

    # Remove any rows with NaN values
    combined_data = pd.concat([y, X], axis=1).dropna()

    y_clean = combined_data[etf_col]
    X_clean = combined_data.drop(columns=[etf_col])
    
    if len(y_clean) == 0:
        print(f"Warning: No valid data for {etf_col}")
        return None, False
    
    # Run regression
    try:
        model = sm.OLS(y_clean, X_clean).fit()
        
        # Test for heteroscedasticity
        _, pvalue_het, _, _ = het_white(model.resid, model.model.exog)
        
        # Use robust standard errors if heteroscedasticity detected
        if pvalue_het < 0.05:
            model_robust = sm.OLS(y_clean, X_clean).fit(cov_type='HC3')
            return model_robust, True
        else:
            return model, False
            
    except Exception as e:
        print(f"Error running regression for {etf_col}: {e}")
        return None, False

# Get actual ETF columns from the regression data
actual_etf_cols = [col for col in regression_data.columns if col in all_etfs]
    
print(f"📊 Available ETF columns: {actual_etf_cols}")

# Run regression for each ETF
regression_results = {}
robust_se_used = {}

for etf in actual_etf_cols:
    print("🔍 Checking regression_data columns:")
    print(regression_data.columns.tolist())
    print(f"🔍 Sample of regression_data:")
    print(regression_data.head())
    print(f"🔍 actual_etf_cols found: {actual_etf_cols}")
    print(f"\n🔄 Processing {etf}...")
    model, robust = run_market_model_regression(regression_data, etf)
    
    if model is not None:
        regression_results[etf] = model
        robust_se_used[etf] = robust
        
        print(f"📈 REGRESSION RESULTS FOR {etf}:")
        print(f"Robust SE used: {robust}")
        print(f"R-squared: {model.rsquared:.4f}")
        print(f"Observations: {model.nobs}")
        
        # Check if event_dummy coefficient exists
        if 'event_dummy' in model.params:
            print(f"Event coefficient: {model.params['event_dummy']:.6f}")
            print(f"Event p-value: {model.pvalues['event_dummy']:.4f}")
            if model.pvalues['event_dummy'] < 0.05:
                print("🔴 SIGNIFICANT EVENT EFFECT DETECTED!")
        
        # Display key coefficients
        key_coeffs = ['event_dummy', 'pre_event', 'post_event', 'us_return']
        for coeff in key_coeffs:
            if coeff in model.params:
                print(f"{coeff}: {model.params[coeff]:.6f} (p={model.pvalues[coeff]:.4f})")
    else:
        print(f"❌ Failed to run regression for {etf}")

print(f"\n✅ Completed regressions for {len(regression_results)} ETFs")


📊 Available ETF columns: ['EWA', 'EWG', 'EWJ', 'EWU', 'EWW', 'EWZ', 'EZA']
🔍 Checking regression_data columns:
['EWA', 'EWG', 'EWJ', 'EWU', 'EWW', 'EWZ', 'EZA', 'us_return', 'event_dummy', 'event_window', 'pre_event', 'post_event', 'vix_change', 'oil_change', 'dollar_change', 'event_pos', 'event_neg']
🔍 Sample of regression_data:
                 EWA       EWG       EWJ       EWU       EWW       EWZ  \
Date                                                                     
2003-02-19  0.767581 -1.783828 -1.687737 -1.822857 -1.444347 -1.866691   
2003-02-20 -1.088144 -0.106836  0.000000  0.088392  0.517254  0.135842   
2003-02-21  0.219996  0.106951  0.143049  0.088324  0.428821  0.000000   
2003-02-24  1.207485 -2.243588 -0.285679 -0.353044 -0.426989  1.628257   
2003-02-25  0.216936 -1.092866 -1.432691 -0.708599 -1.457999 -2.269674   

                 EZA  us_return  event_dummy  event_window  pre_event  \
Date                                                                    
200

In [20]:
# =============================================================================
# STEP 6: CALCULATE ABNORMAL RETURNS
# =============================================================================

def calculate_abnormal_returns(data, model_results, etf_col):
    """
    Calculate abnormal returns using the fitted market model.
    Normal returns are predicted with event-related dummies set to 0.
    """
    model = model_results[etf_col]

    # Start with same index as data
    X_normal = pd.DataFrame(index=data.index)

    # Core regressors
    X_normal['us_return']   = data['us_return']
    X_normal['event_pos'] = 0.0  
    X_normal['event_neg'] = 0.0       # turn off event effect
    X_normal['pre_event']   = 0.0
    X_normal['post_event']  = 0.0

    # Controls (only if present in model)
    for var in ['vix_change', 'oil_change', 'dollar_change']:
        if var in data.columns and var in model.params.index:
            X_normal[var] = data[var]

    # Add constant with the same name used in estimation ('const')
    X_normal = sm.add_constant(X_normal, has_constant='add')

    # Keep only columns that the model actually used (and in same order)
    X_normal = X_normal[model.params.index]

    # Predict normal returns
    normal_returns = model.predict(X_normal)

    # Abnormal = actual - normal
    abnormal_returns = data[etf_col] - normal_returns

    return abnormal_returns


# Calculate abnormal returns for each ETF you actually estimated
abnormal_returns = {}
for etf in regression_results.keys():
    abnormal_returns[etf] = calculate_abnormal_returns(
        regression_data, regression_results, etf
    )

print("📊 ABNORMAL RETURNS CALCULATED FOR ALL ESTIMATED ETFs")


📊 ABNORMAL RETURNS CALCULATED FOR ALL ESTIMATED ETFs


In [21]:
rows = []
for etf, model in regression_results.items():
    row = {
        'ETF': etf,
        'beta_us': model.params.get('us_return', np.nan),
        'beta_us_p': model.pvalues.get('us_return', np.nan),
        'event_pos_coef': model.params.get('event_pos', np.nan),
        'event_pos_p': model.pvalues.get('event_pos', np.nan),
        'event_neg_coef': model.params.get('event_neg', np.nan),
        'event_neg_p': model.pvalues.get('event_neg', np.nan),
        'pre_coef': model.params.get('pre_event', np.nan),
        'post_coef': model.params.get('post_event', np.nan),
        'R2': model.rsquared,
        'N': int(model.nobs),
    }
    rows.append(row)

reg_table = pd.DataFrame(rows)
print(reg_table)


   ETF   beta_us      beta_us_p  event_pos_coef  event_pos_p  event_neg_coef  \
0  EWA  0.984926  6.102953e-167        3.995154     0.032066       -1.753959   
1  EWG  0.950883   0.000000e+00        1.933624     0.415933       -0.709381   
2  EWJ  0.711436  8.270546e-237        1.708389     0.395757       -0.697034   
3  EWU  0.844914  2.410557e-296        1.913221     0.411945       -2.394503   
4  EWW  0.910611  1.771273e-184        1.748554     0.547323        0.302339   
5  EWZ  1.088705   2.524522e-83        3.489306     0.294490       -5.682565   
6  EZA  1.019925  1.047570e-109        1.413859     0.638643       -3.052265   

   event_neg_p  pre_coef  post_coef        R2     N  
0     0.110989 -0.720403   0.115719  0.688416  5754  
1     0.264353 -0.199881  -0.023748  0.755259  5754  
2     0.542981 -0.267102   0.210493  0.557936  5754  
3     0.001958 -0.734322   0.237399  0.732439  5754  
4     0.796232 -0.426039  -0.690928  0.547180  5754  
5     0.000837  0.715085  -0.199188

In [ ]:
import pandas as pd
import numpy as np

# Window in calendar days
LAG_PRE = -5
LAG_POST = 10

def build_event_time_panel(abnormal_returns, event_dates, lag_pre=-5, lag_post=10):
    """
    Returns a long DataFrame with columns:
    ['ETF', 'event_date', 't', 'abnormal_ret', 'car']
    where t is event time (0 = event day).
    """
    rows = []

    for etf, ar_series in abnormal_returns.items():
        for ev in event_dates:
            start = ev + pd.Timedelta(days=lag_pre)
            end   = ev + pd.Timedelta(days=lag_post)

            # Slice abnormal returns for this ETF in calendar time
            window_ar = ar_series.loc[start:end]

            if window_ar.empty:
                continue

            # Align to event time t = 0 on the event date
            for dt, ar in window_ar.items():
                t = (dt - ev).days   # event time in days
                rows.append({
                    'ETF': etf,
                    'event_date': ev,
                    'date': dt,
                    't': t,
                    'abnormal_ret': ar
                })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # Compute CAR by ETF, event, and t
    df = df.sort_values(['ETF', 'event_date', 't'])
    df['car'] = df.groupby(['ETF', 'event_date'])['abnormal_ret'].cumsum()

    return df

event_panel = build_event_time_panel(abnormal_returns, event_dates,
                                     lag_pre=LAG_PRE, lag_post=LAG_POST)
print(event_panel.head())


developed = ['EWA', 'EWG', 'EWJ', 'EWU']
emerging  = ['EWZ', 'EWW', 'EZA']

event_panel['group'] = np.where(event_panel['ETF'].isin(developed),
                                'Developed', 'Emerging')

   ETF event_date       date  t  abnormal_ret       car
0  EWA 2008-09-29 2008-09-24 -5      1.727843  1.727843
1  EWA 2008-09-29 2008-09-25 -4     -0.372216  1.355627
2  EWA 2008-09-29 2008-09-26 -3     -2.188859 -0.833232
3  EWA 2008-09-29 2008-09-29  0     -2.758101 -3.591333
4  EWA 2008-09-29 2008-09-30  1      0.240215 -3.351118


In [ ]:
avg_car_etf = (
    event_panel
    .groupby(['ETF', 't'], as_index=False)
    .agg(avg_car=('car', 'mean'))
)

avg_car_group = (
    event_panel
    .groupby(['group', 't'], as_index=False)
    .agg(avg_car=('car', 'mean'))
)

import plotly.express as px

fig_etf = px.line(
    avg_car_etf,
    x='t', y='avg_car',
    color='ETF',
    title='Average Cumulative Abnormal Returns Around US Events',
    labels={'t': 'Event Time (days)', 'avg_car': 'Average CAR (%)'}
)
fig_etf.add_vline(x=0, line_dash='dash', line_color='black')  # event day
fig_etf.show()


In [ ]:
fig_group = px.line(
    avg_car_group,
    x='t', y='avg_car',
    color='group',
    title='Average CAR Around US Events: Developed vs Emerging ETFs',
    labels={'t': 'Event Time (days)', 'avg_car': 'Average CAR (%)'}
)
fig_group.add_vline(x=0, line_dash='dash', line_color='black')
fig_group.show()


In [ ]:
event_day_ar = (
    event_panel[event_panel['t'] == 0]
    .groupby('ETF', as_index=False)
    .agg(mean_ar=('abnormal_ret', 'mean'))
)

fig_bar = px.bar(
    event_day_ar,
    x='ETF', y='mean_ar',
    title='Mean Abnormal Return on Event Days',
    labels={'mean_ar': 'Mean Abnormal Return (%)'}
)
fig_bar.show()


In [18]:
# Make sure event_dates are Timestamps matching us_returns.index
event_dates = [
    pd.Timestamp('2008-09-29'),
    pd.Timestamp('2008-10-13'),
    pd.Timestamp('2008-10-15'),
    pd.Timestamp('2008-10-28'),
    pd.Timestamp('2008-12-01'),
    pd.Timestamp('2020-03-12'),
    pd.Timestamp('2020-03-13'),
    pd.Timestamp('2020-03-16'),
    pd.Timestamp('2020-03-24'),
    pd.Timestamp('2025-04-09'),
]

event_info = []
for d in event_dates:
    r = float(us_returns.loc[d,'^GSPC' ])
    event_info.append({
        'event_date': d,
        'us_return': r,
        'type': 'positive' if r > 0 else 'negative'
    })

event_df = pd.DataFrame(event_info)
print(event_df)


  event_date  us_return      type
0 2008-09-29  -8.806776  negative
1 2008-10-13  11.580037  positive
2 2008-10-15  -9.034978  negative
3 2008-10-28  10.789006  positive
4 2008-12-01  -8.929524  negative
5 2020-03-12  -9.511268  negative
6 2020-03-13   9.287125  positive
7 2020-03-16 -11.984055  negative
8 2020-03-24   9.382774  positive
9 2025-04-09   9.515388  positive


In [17]:
LAG_PRE = -5
LAG_POST = 10

def build_event_time_panel_with_type(abnormal_returns, event_df, lag_pre=-5, lag_post=10):
    rows = []

    for etf, ar_series in abnormal_returns.items():
        for _, row in event_df.iterrows():
            ev = row['event_date']
            etype = row['type']  # 'positive' or 'negative'

            start = ev + pd.Timedelta(days=lag_pre)
            end   = ev + pd.Timedelta(days=lag_post)

            window_ar = ar_series.loc[start:end]
            if window_ar.empty:
                continue

            for dt, ar in window_ar.items():
                t = (dt - ev).days
                rows.append({
                    'ETF': etf,
                    'event_date': ev,
                    'event_type': etype,
                    'date': dt,
                    't': t,
                    'abnormal_ret': ar
                })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df = df.sort_values(['ETF', 'event_date', 't'])
    df['car'] = df.groupby(['ETF', 'event_date'])['abnormal_ret'].cumsum()

    return df

event_panel = build_event_time_panel_with_type(abnormal_returns, event_df,
                                               lag_pre=LAG_PRE, lag_post=LAG_POST)
print(event_panel.head())

developed = ['EWA', 'EWG', 'EWJ', 'EWU']
emerging  = ['EWZ', 'EWW', 'EZA']

event_panel['group'] = np.where(event_panel['ETF'].isin(developed),
                                'Developed', 'Emerging')


NameError: name 'abnormal_returns' is not defined

In [ ]:
avg_car_group_type = (
    event_panel
    .groupby(['group', 'event_type', 't'], as_index=False)
    .agg(avg_car=('car', 'mean'))
)

import plotly.express as px

fig = px.line(
    avg_car_group_type,
    x='t',
    y='avg_car',
    color='group',
    facet_col='event_type',       # one panel for positive, one for negative
    facet_col_spacing=0.04,
    title='Average CAR Around US Events, by Event Type and Group',
    labels={'t': 'Event Time (days)', 'avg_car': 'Average CAR (%)'}
)

# Add a vertical line at t=0 in each facet
fig.add_vline(x=0, line_dash='dash', line_color='black')

fig.update_layout(showlegend=True)
fig.show()


In [ ]:
avg_car_etf_type = (
    event_panel
    .groupby(['ETF', 'event_type', 't'], as_index=False)
    .agg(avg_car=('car', 'mean'))
)

fig_etf = px.line(
    avg_car_etf_type,
    x='t',
    y='avg_car',
    color='ETF',
    facet_col='event_type',
    title='Average CAR by ETF and Event Type',
    labels={'t': 'Event Time (days)', 'avg_car': 'Average CAR (%)'}
)
fig_etf.add_vline(x=0, line_dash='dash', line_color='black')
fig_etf.show()


In [ ]:
# t = 0 rows only
event_day_ar = (
    event_panel[event_panel['t'] == 0]
    .groupby(['ETF', 'event_type'], as_index=False)
    .agg(mean_ar=('abnormal_ret', 'mean'))
)
print(event_day_ar)

import plotly.express as px

fig_bar = px.bar(
    event_day_ar,
    x='ETF',
    y='mean_ar',
    color='event_type',
    barmode='group',
    title='Mean Abnormal Return on Event Days by Event Type',
    labels={
        'ETF': 'ETF',
        'mean_ar': 'Mean Abnormal Return (%)',
        'event_type': 'US Event Type'
    }
)
fig_bar.add_hline(y=0, line_dash='dash', line_color='black')
fig_bar.show()


    ETF event_type   mean_ar
0   EWA   negative -1.465465
1   EWA   positive  3.299303
2   EWG   negative -0.576916
3   EWG   positive  1.629653
4   EWJ   negative -0.508603
5   EWJ   positive  1.500146
6   EWU   negative -2.155410
7   EWU   positive  1.371883
8   EWW   negative  0.042917
9   EWW   positive  1.181348
10  EWZ   negative -4.999654
11  EWZ   positive  2.967091
12  EZA   negative -2.697819
13  EZA   positive  1.291707


In [ ]:
reg_table.to_latex('regression_results.tex', index=False, float_format="%.3f")


OSError: [Errno 30] Read-only file system: 'regression_results.tex'

In [22]:
latex_str = reg_table.to_latex(index=False, float_format="%.3f")
print(latex_str)


\begin{tabular}{lrrrrrrrrrr}
\toprule
ETF & beta_us & beta_us_p & event_pos_coef & event_pos_p & event_neg_coef & event_neg_p & pre_coef & post_coef & R2 & N \\
\midrule
EWA & 0.985 & 0.000 & 3.995 & 0.032 & -1.754 & 0.111 & -0.720 & 0.116 & 0.688 & 5754 \\
EWG & 0.951 & 0.000 & 1.934 & 0.416 & -0.709 & 0.264 & -0.200 & -0.024 & 0.755 & 5754 \\
EWJ & 0.711 & 0.000 & 1.708 & 0.396 & -0.697 & 0.543 & -0.267 & 0.210 & 0.558 & 5754 \\
EWU & 0.845 & 0.000 & 1.913 & 0.412 & -2.395 & 0.002 & -0.734 & 0.237 & 0.732 & 5754 \\
EWW & 0.911 & 0.000 & 1.749 & 0.547 & 0.302 & 0.796 & -0.426 & -0.691 & 0.547 & 5754 \\
EWZ & 1.089 & 0.000 & 3.489 & 0.294 & -5.683 & 0.001 & 0.715 & -0.199 & 0.476 & 5754 \\
EZA & 1.020 & 0.000 & 1.414 & 0.639 & -3.052 & 0.096 & 0.829 & -0.267 & 0.560 & 5754 \\
\bottomrule
\end{tabular}



In [24]:
reg_table.to_latex('/Users/KarenKings/Documents/University/WINTER 2026/ECON 559/Project/regression_results.tex',
                   index=False,
                   float_format="%.3f")
